In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# CHEBYSHEV TYPE II LOW-PASS FILTER DESIGN EXERCISE
#
# Specifications:
#
#       ωp = 0.5 rad/s
#       ωs = 1.0 rad/s
#       Ap = 0.5 dB
#       As = 30.0 dB
#
# Design procedure:
#
# 1. Calculate ε.
# 2. Calculate the minimum order N.
# 3. Calculate the stable poles from the Chebyshev-II pole equations.
# 4. Calculate the finite transmission zeros.
# 5. Construct the pole and zero factors.
# 6. Calculate H0 from the condition H(0) = 1.
# 7. Construct H(s).
# 8. Construct H(jω).
# 9. Calculate magnitude, phase and group delay.
#
# No numerical result from the printed analytical solution is hard-coded.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# FILTER SPECIFICATIONS
# ==============================================================================

wp = 0.5
ws = 1.0
Ap = 0.5
As = 30.0

# ==============================================================================
# STEP 1: RIPPLE PARAMETER
#
# ε = 1 / sqrt(10^(As/10) - 1)
# ==============================================================================

epsilon = 1.0 / np.sqrt(10.0**(As / 10.0) - 1.0)

# ==============================================================================
# STEP 2: MINIMUM FILTER ORDER
#
# N = acosh[
#       sqrt((10^(As/10)-1)/(10^(Ap/10)-1))
#     ] / acosh(ωs/ωp)
# ==============================================================================

N_exact = np.arccosh(np.sqrt((10.0**(As / 10.0) - 1.0) / (10.0**(Ap / 10.0) - 1.0))) / np.arccosh(ws / wp)

N = int(np.ceil(N_exact))

# ==============================================================================
# STEP 3: AUXILIARY PARAMETERS
# ==============================================================================

asinh_term = np.arcsinh(1.0 / epsilon)

alpha = asinh_term / N

sinh_alpha = np.sinh(alpha)

cosh_alpha = np.cosh(alpha)

# ==============================================================================
# STEP 4: STABLE CHEBYSHEV-II POLES
#
# θk = (2k-1)π/(2N)
#
# Dk = sinh²(α) sin²(θk) + cosh²(α) cos²(θk)
#
# σk = -ωs sinh(α) sin(θk) / Dk
#
# ωk = -ωs cosh(α) cos(θk) / Dk
#
# k = 1,...,N
#
# These N values are directly the stable left-half-plane poles.
# ==============================================================================

poles_numeric = []

for k in range(1, N + 1):

    theta_k = (2.0 * k - 1.0) * np.pi / (2.0 * N)

    D_k = sinh_alpha**2 * np.sin(theta_k)**2 + cosh_alpha**2 * np.cos(theta_k)**2

    sigma_k = -ws * sinh_alpha * np.sin(theta_k) / D_k

    omega_k = -ws * cosh_alpha * np.cos(theta_k) / D_k

    p_k = sigma_k + 1j * omega_k

    poles_numeric.append(p_k)

poles_numeric = np.array(poles_numeric)

# ==============================================================================
# STEP 5: FINITE TRANSMISSION ZEROS
#
# zk = ±j ωs / cos[(2k-1)π/(2N)]
#
# For even N there are N finite zeros.
# ==============================================================================

zeros_numeric = []

for k in range(1, N // 2 + 1):

    theta_z = (2.0 * k - 1.0) * np.pi / (2.0 * N)

    omega_z = ws / np.cos(theta_z)

    zeros_numeric.append(1j * omega_z)

    zeros_numeric.append(-1j * omega_z)

zeros_numeric = np.array(zeros_numeric)

# ==============================================================================
# SYMBOLIC VARIABLES
# ==============================================================================

s = sp.symbols('s', real=True)
omega = sp.symbols('omega', real=True)
I = sp.I

# ==============================================================================
# STEP 6: CONVERT COMPUTED POLES AND ZEROS TO SYMBOLIC NUMBERS
# ==============================================================================

poles_symbolic = []

for p_k in poles_numeric:

    poles_symbolic.append(sp.Float(p_k.real, 16) + I * sp.Float(p_k.imag, 16))

zeros_symbolic = []

for z_k in zeros_numeric:

    zeros_symbolic.append(sp.Float(z_k.real, 16) + I * sp.Float(z_k.imag, 16))

# ==============================================================================
# STEP 7: SYMBOLIC DENOMINATOR
#
# D(s) = Π(s - pk)
# ==============================================================================

denominator_symbolic = sp.Integer(1)

for p_k in poles_symbolic:

    denominator_symbolic *= (s - p_k)

denominator_symbolic = sp.expand(denominator_symbolic)

denominator_symbolic = sp.N(sp.re(denominator_symbolic), 12)

# ==============================================================================
# STEP 8: SYMBOLIC ZERO POLYNOMIAL
#
# Z(s) = Π(s - zk)
# ==============================================================================

zero_polynomial_symbolic = sp.Integer(1)

for z_k in zeros_symbolic:

    zero_polynomial_symbolic *= (s - z_k)

zero_polynomial_symbolic = sp.expand(zero_polynomial_symbolic)

zero_polynomial_symbolic = sp.N(sp.re(zero_polynomial_symbolic), 12)

# ==============================================================================
# STEP 9: NORMALIZATION CONSTANT H0
#
# For even-order Chebyshev-II filters:
#
# H0 = product(|pk|²) / product(ωzk²)
#
# where one member from each conjugate pole/zero pair is used.
#
# Equivalently, because H(0) = 1:
#
# H0 = D(0) / Z(0)
#
# We calculate it symbolically from the constructed polynomials.
# ==============================================================================

H0_symbolic = sp.N(denominator_symbolic.subs(s, 0) / zero_polynomial_symbolic.subs(s, 0), 12)

# ==============================================================================
# STEP 10: TRANSFER FUNCTION
# ==============================================================================

numerator_symbolic = sp.expand(H0_symbolic * zero_polynomial_symbolic)

H_s = sp.cancel(numerator_symbolic / denominator_symbolic)

# ==============================================================================
# STEP 11: POLE AND ZERO FACTORS FOR DISPLAY
# ==============================================================================

upper_poles = []

for p_k in poles_numeric:

    if p_k.imag > 1e-10:

        upper_poles.append(p_k)

upper_zeros = []

for z_k in zeros_numeric:

    if z_k.imag > 1e-10:

        upper_zeros.append(z_k)

pole_factors = []

for p_k in upper_poles:

    sigma = sp.Float(p_k.real, 16)

    omega_k = sp.Float(p_k.imag, 16)

    factor_k = sp.expand(s**2 - 2.0 * sigma * s + sigma**2 + omega_k**2)

    pole_factors.append(sp.N(factor_k, 10))

zero_factors = []

for z_k in upper_zeros:

    omega_z = sp.Float(z_k.imag, 16)

    factor_k = sp.expand(s**2 + omega_z**2)

    zero_factors.append(sp.N(factor_k, 10))

# ==============================================================================
# STEP 12: FREQUENCY RESPONSE
#
# s -> jω
# ==============================================================================

numerator_jw = sp.expand(numerator_symbolic.subs(s, I * omega))

denominator_jw = sp.expand(denominator_symbolic.subs(s, I * omega))

num_real = sp.N(sp.re(numerator_jw), 12)

num_imag = sp.N(sp.im(numerator_jw), 12)

den_real = sp.N(sp.re(denominator_jw), 12)

den_imag = sp.N(sp.im(denominator_jw), 12)

H_jw = sp.cancel(numerator_jw / denominator_jw)

# ==============================================================================
# STEP 13: SYMBOLIC MAGNITUDE RESPONSE
# ==============================================================================

num_power = sp.expand(num_real**2 + num_imag**2)

den_power = sp.expand(den_real**2 + den_imag**2)

magnitude_squared_symbolic = sp.cancel(num_power / den_power)

magnitude_symbolic = sp.sqrt(magnitude_squared_symbolic)

# ==============================================================================
# STEP 14: SYMBOLIC PHASE RESPONSE
#
# φH = arg N - arg D
#
# For display we retain the numerator and denominator arguments separately.
# ==============================================================================

numerator_phase_ratio = sp.cancel(num_imag / num_real)

denominator_phase_ratio = sp.cancel(den_imag / den_real)

# ==============================================================================
# STEP 15: SYMBOLIC GROUP DELAY
#
# For F(jω) = R + jI:
#
# d(arg F)/dω = (R I' - I R') / (R² + I²)
#
# Since:
#
# arg H = arg N - arg D
#
# τ = -d(arg H)/dω
#
#   = d(arg D)/dω - d(arg N)/dω
# ==============================================================================

num_real_derivative = sp.diff(num_real, omega)

num_imag_derivative = sp.diff(num_imag, omega)

den_real_derivative = sp.diff(den_real, omega)

den_imag_derivative = sp.diff(den_imag, omega)

numerator_phase_derivative = sp.cancel((num_real * num_imag_derivative - num_imag * num_real_derivative) / (num_real**2 + num_imag**2))

denominator_phase_derivative = sp.cancel((den_real * den_imag_derivative - den_imag * den_real_derivative) / (den_real**2 + den_imag**2))

group_delay_symbolic = sp.cancel(denominator_phase_derivative - numerator_phase_derivative)

group_delay_symbolic = sp.together(group_delay_symbolic)

# ==============================================================================
# NUMERICAL FUNCTIONS FOR PLOTTING
# ==============================================================================

num_real_function = sp.lambdify(omega, num_real, 'numpy')

num_imag_function = sp.lambdify(omega, num_imag, 'numpy')

den_real_function = sp.lambdify(omega, den_real, 'numpy')

den_imag_function = sp.lambdify(omega, den_imag, 'numpy')

group_delay_function = sp.lambdify(omega, group_delay_symbolic, 'numpy')

# ==============================================================================
# FREQUENCY AXIS
# ==============================================================================

omega_values = np.logspace(-2, 2, 5000)

# ==============================================================================
# NUMERICAL H(jω)
# ==============================================================================

num_real_values = np.asarray(num_real_function(omega_values), dtype=float)

num_imag_values = np.asarray(num_imag_function(omega_values), dtype=float)

den_real_values = np.asarray(den_real_function(omega_values), dtype=float)

den_imag_values = np.asarray(den_imag_function(omega_values), dtype=float)

if num_real_values.ndim == 0:

    num_real_values = np.full_like(omega_values, float(num_real_values))

if num_imag_values.ndim == 0:

    num_imag_values = np.full_like(omega_values, float(num_imag_values))

H_values = (num_real_values + 1j * num_imag_values) / (den_real_values + 1j * den_imag_values)

# ==============================================================================
# MAGNITUDE VALUES
# ==============================================================================

magnitude_values = np.abs(H_values)

# ==============================================================================
# PHASE VALUES
# ==============================================================================

phase_values = np.unwrap(np.angle(H_values))

phase_deg_values = np.rad2deg(phase_values)

# ==============================================================================
# GROUP-DELAY VALUES
# ==============================================================================

group_delay_values = np.asarray(group_delay_function(omega_values), dtype=float)

if group_delay_values.ndim == 0:

    group_delay_values = np.full_like(omega_values, float(group_delay_values))

# Do not draw numerical infinities exactly at transmission zeros
group_delay_values[~np.isfinite(group_delay_values)] = np.nan

# ==============================================================================
# TEXT: POLES
# ==============================================================================

pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.6f} {p.imag:+.6f}j' for k, p in enumerate(poles_numeric)])

# ==============================================================================
# TEXT: ZEROS
# ==============================================================================

zero_text = '<br>'.join([f'z{k + 1} = {z.real:+.6f} {z.imag:+.6f}j' for k, z in enumerate(zeros_numeric)])

# ==============================================================================
# TEXT: FACTORS
# ==============================================================================

pole_factor_text = ''

for index, factor_k in enumerate(pole_factors):

    pole_factor_text += f'Pole pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

zero_factor_text = ''

for index, factor_k in enumerate(zero_factors):

    zero_factor_text += f'Zero pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

# ==============================================================================
# NUMERICAL FORMS FOR DISPLAY
# ==============================================================================

H0_display = float(H0_symbolic)

numerator_display = sp.N(numerator_symbolic, 8)

denominator_display = sp.N(denominator_symbolic, 8)

num_real_display = sp.N(num_real, 8)

num_imag_display = sp.N(num_imag, 8)

den_real_display = sp.N(den_real, 8)

den_imag_display = sp.N(den_imag, 8)

magnitude_display = sp.N(magnitude_symbolic, 8)

group_delay_display = sp.N(group_delay_symbolic, 7)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1240px;
    max-width:1240px;
    box-sizing:border-box;
">
<b>Chebyshev Type II Design Exercise</b><br>
Construct a Chebyshev Type II low-pass filter with
ω<sub>p</sub> = {wp:.1f} rad/s,
ω<sub>s</sub> = {ws:.1f} rad/s,
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.0f} dB.
<br>
<b>Purpose:</b>
Follow the analytical Chebyshev-II design procedure. The ripple parameter,
minimum order, poles, transmission zeros and all filter coefficients are
computed from the theoretical equations; no numerical result from the printed
solution is hard-coded.
</div>
""", layout=Layout(width='1250px', max_width='1250px'))

# ==============================================================================
# INFORMATION PANEL
# ==============================================================================

info_html = HTML(f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:10px 11px;
    font-size:12px;
    line-height:1.58;
    background:white;
    width:590px;
    box-sizing:border-box;
">

<b>Step 1 — Filter specifications</b><br>
<span style="color:#0066cc;">
ωp = {wp:.1f} rad/s,
ωs = {ws:.1f} rad/s,
Ap = {Ap:.1f} dB,
As = {As:.0f} dB
</span>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 2 — Ripple parameter and minimum order</b><br>
ε = <span style="color:#0066cc;">{epsilon:.6f}</span><br>
Nmin = <span style="color:#0066cc;">{N_exact:.6f}</span><br>
N = <span style="color:#0066cc;"><b>{N}</b></span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 3 — Auxiliary hyperbolic quantities</b><br>
sinh⁻¹(1/ε) = <span style="color:#0066cc;">{asinh_term:.6f}</span><br>
α = (1/N)sinh⁻¹(1/ε) = <span style="color:#0066cc;">{alpha:.6f}</span><br>
sinh(α) = <span style="color:#0066cc;">{sinh_alpha:.6f}</span><br>
cosh(α) = <span style="color:#0066cc;">{cosh_alpha:.6f}</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 4 — Transmission zeros from the theoretical equations</b><br>
<span style="color:#0066cc;">
{zero_text}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 5 — Stable poles from the theoretical equations</b><br>
<span style="color:#0066cc;">
{pole_text}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 6 — Zero factors</b><br>
{zero_factor_text}
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 7 — Pole factors</b><br>
{pole_factor_text}
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 8 — Normalization constant</b><br>
H₀ = <span style="color:#0066cc;">{H0_display:.6f}</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 9 — Transfer function</b><br>
H(s) = N(s) / D(s)<br>
<span style="color:#0066cc;">
N(s) = {sp.sstr(numerator_display)}
</span><br>
<span style="color:#0066cc;">
D(s) = {sp.sstr(denominator_display)}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 10 — Frequency response</b><br>
N(jω) =
<span style="color:#0066cc;">
({sp.sstr(num_real_display)}) + j({sp.sstr(num_imag_display)})
</span><br>

D(jω) =
<span style="color:#0066cc;">
({sp.sstr(den_real_display)}) + j({sp.sstr(den_imag_display)})
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 11 — Magnitude response</b><br>
<span style="color:#0066cc;">
|H(jω)| = {sp.sstr(magnitude_display)}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 12 — Group delay</b><br>
τ(ω) =
<span style="color:#0066cc;">
{sp.sstr(group_delay_display)}
</span>
</div>

</div>
""", layout=Layout(width='600px', max_width='600px'))

# ==============================================================================
# COMMON FIGURE SETTINGS
# ==============================================================================

title_fontsize = 11
label_fontsize = 9
tick_fontsize = 8
legend_fontsize = 8

# ==============================================================================
# FIGURE 1: MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.3, 3.0))

ax_mag.plot(omega_values, magnitude_values, 'r-', linewidth=2.0, label='|H(jω)|')

ax_mag.axvline(ws, color='black', linestyle=':', linewidth=1.0, label='ωs')

ax_mag.axhline(10.0**(-As / 20.0), color='gray', linestyle='--', linewidth=0.9, label='Stopband limit')

ax_mag.set_xscale('log')
ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_mag.set_ylabel('|H(jω)|', fontsize=label_fontsize)
ax_mag.set_title('Chebyshev II Magnitude Response', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_mag.tick_params(axis='both', labelsize=tick_fontsize)
ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)
ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=3, fontsize=legend_fontsize)

ax_mag.set_xlim(0.01, 100.0)
ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False
fig_mag.canvas.layout.width = '530px'
fig_mag.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.3, 3.0))

ax_phase.plot(omega_values, phase_deg_values, 'r-', linewidth=2.0, label='∠H(jω)')

ax_phase.axvline(ws, color='black', linestyle=':', linewidth=1.0, label='ωs')

ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_phase.set_xscale('log')
ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_phase.set_ylabel('Phase (degrees)', fontsize=label_fontsize)
ax_phase.set_title('Chebyshev II Phase Response', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_phase.tick_params(axis='both', labelsize=tick_fontsize)
ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)
ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=2, fontsize=legend_fontsize)

ax_phase.set_xlim(0.01, 100.0)

fig_phase.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False
fig_phase.canvas.layout.width = '530px'
fig_phase.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(5.3, 3.0))

ax_gd.plot(omega_values, group_delay_values, 'r-', linewidth=2.0, label='τ(ω)')

ax_gd.axvline(ws, color='black', linestyle=':', linewidth=1.0, label='ωs')

ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xscale('log')
ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)
ax_gd.set_ylabel('Group Delay τ(ω)', fontsize=label_fontsize)
ax_gd.set_title('Chebyshev II Group Delay', fontsize=title_fontsize, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=tick_fontsize)
ax_gd.grid(True, which='both', linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=2, fontsize=legend_fontsize)

ax_gd.set_xlim(0.01, 100.0)

# Fixed visible range. Singular behavior at transmission zeros is intentionally
# clipped so the finite group-delay behavior remains readable.
ax_gd.set_ylim(-10.0, 10.0)

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '530px'
fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# LAYOUT
# ==============================================================================

left_column = VBox([info_html], layout=Layout(width='610px', min_width='610px', max_width='610px', flex='0 0 610px', align_items='flex-start'))

right_column = VBox([fig_mag.canvas, fig_phase.canvas, fig_gd.canvas], layout=Layout(width='540px', min_width='540px', max_width='540px', flex='0 0 540px', align_items='flex-start'))

main_layout = HBox([left_column, right_column], layout=Layout(width='1160px', min_width='1160px', max_width='1160px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)